# Run Full ICD-10 Coding Experiment
Consolidated pipeline: extraction → search → matching → evaluation.  
Assumes UDFs and setup from notebooks 00-04 are already deployed.

*Co-authored with CoCo*

In [ ]:
%%sql -r params_set
-- ============================================================
-- CONFIGURABLE PARAMETERS
-- ============================================================

-- Models
SET EXTRACTION_MODEL = 'claude-4-sonnet';
SET MATCHING_MODEL   = 'claude-4-sonnet';
SET EVAL_MODEL       = 'claude-4-sonnet';

-- Extraction
SET MAX_DX_SECTION_LEN = 8000;
SET MAX_HPI_LEN        = 4000;
SET MAX_PMH_LEN        = 2000;

-- Search
SET SEMANTIC_K       = 10;
SET CATEGORY_K       = 5;
SET HCC_EXPANSION_K  = 5;
SET FINAL_TOP_N      = 15;
SET WEIGHT_SEMANTIC  = 1.0;
SET WEIGHT_CATEGORY  = 0.7;
SET WEIGHT_HCC       = 0.5;

-- Matching
SET CONFIDENCE_THRESHOLD = 0.75;

-- Experiment metadata
SET RUN_ID = 'eval_v2_' || TO_CHAR(CURRENT_TIMESTAMP(), 'YYYYMMDD_HH24MISS');
SET STRATEGY = 'FULL_PIPELINE_V2';

-- Extraction prompt
SET EXTRACTION_PROMPT = '
You are a certified ICD-10-CM coder performing chart abstraction. Extract every codeable clinical condition from this encounter note.

For EACH condition, return a JSON object with these EXACT keys:
{
  "finding": "<clinical term as stated by clinician>",
  "category": "diagnosis|symptom|history_of|ruled_out|suspected",
  "acuity": "acute|chronic|acute_on_chronic|unspecified",
  "laterality": "left|right|bilateral|unspecified|not_applicable",
  "body_site": "<specific anatomical site or null>",
  "severity": "mild|moderate|severe|unspecified",
  "causal_link": "<underlying cause if explicitly stated, or null>",
  "supporting_quote": "<verbatim text from note>",
  "is_present_on_admission": true|false|null
}

CODING RULES:
1. "History of" is NOT an active condition. Mark category="history_of".
2. If a condition has a stated cause (e.g. "diabetic nephropathy"), populate causal_link.
3. Extract EVERY laterality and body site. "Right knee OA" needs laterality=right, body_site=knee.
4. If note says "chronic" or duration >6 weeks, mark acuity=chronic.
5. Do NOT extract vitals, labs, meds, procedures UNLESS they ARE a diagnosis.
6. supporting_quote MUST be a verbatim substring from the document.
7. Return ONLY a JSON array. No markdown fences, no explanation.
';

-- Matching prompt
SET MATCHING_PROMPT = '
You are a certified medical coder. Select the BEST ICD-10-CM code for this clinical finding from the candidates.
Consider specificity, laterality, acuity, and clinical context.
If no candidate is appropriate, respond with NONE.
Respond ONLY with: {"code": "<ICD-10>", "description": "<desc>", "confidence": <0.0-1.0>, "rationale": "<brief>"}
';

In [ ]:
%%sql -r extraction_result
SET QUERY_TAG_VAL = 'experiment:extraction|run:' || $RUN_ID;
ALTER SESSION SET QUERY_TAG = $QUERY_TAG_VAL;

USE DATABASE ICD10_CODING_APP;
USE SCHEMA PROCESSING;

CREATE OR REPLACE TABLE PROCESSING.ENCOUNTER_DIAGNOSES_RAW AS
SELECT
    FILE_NAME,
    EXTRACT_CLINICAL_FINDINGS(DIAGNOSIS_SECTION, HPI_SECTION, PMH_SECTION, ENCOUNTER_SETTING) AS RAW_RESPONSE,
    CURRENT_TIMESTAMP() AS EXTRACTED_AT
FROM PROCESSING.DOCUMENT_SECTIONS;

CREATE OR REPLACE TABLE PROCESSING.ENCOUNTER_FINDINGS AS
SELECT
    r.FILE_NAME,
    ROW_NUMBER() OVER (PARTITION BY r.FILE_NAME ORDER BY f.INDEX) AS FINDING_SEQ,
    TRIM(f.VALUE:finding::VARCHAR) AS FINDING,
    LOWER(TRIM(f.VALUE:category::VARCHAR)) AS CATEGORY,
    LOWER(TRIM(f.VALUE:acuity::VARCHAR)) AS ACUITY,
    LOWER(TRIM(f.VALUE:laterality::VARCHAR)) AS LATERALITY,
    TRIM(f.VALUE:body_site::VARCHAR) AS BODY_SITE,
    LOWER(TRIM(f.VALUE:severity::VARCHAR)) AS SEVERITY,
    TRIM(f.VALUE:causal_link::VARCHAR) AS CAUSAL_LINK,
    TRIM(f.VALUE:supporting_quote::VARCHAR) AS SUPPORTING_QUOTE,
    f.VALUE:is_present_on_admission::BOOLEAN AS IS_POA,
    r.EXTRACTED_AT
FROM PROCESSING.ENCOUNTER_DIAGNOSES_RAW r,
    LATERAL FLATTEN(INPUT => TRY_PARSE_JSON(
        REGEXP_REPLACE(r.RAW_RESPONSE, '^```(json)?\\s*|\\s*```$', '')
    )) f
WHERE f.VALUE:finding IS NOT NULL;

In [ ]:
%%sql -r search_result
SET QUERY_TAG_VAL = 'experiment:search|run:' || $RUN_ID;
ALTER SESSION SET QUERY_TAG = $QUERY_TAG_VAL;

USE DATABASE ICD10_CODING_APP;
USE SCHEMA MATCHING;

CREATE OR REPLACE TABLE MATCHING.SEARCH_QUERIES AS
SELECT
    FILE_NAME, FINDING_SEQ, FINDING, CATEGORY, ACUITY, LATERALITY, BODY_SITE, SEVERITY, CAUSAL_LINK, SUPPORTING_QUOTE,
    BUILD_SEARCH_QUERY(FINDING, CATEGORY, ACUITY, LATERALITY, BODY_SITE, SEVERITY, CAUSAL_LINK) AS SEARCH_QUERY,
    GET_CHAPTER_PREFIX(FINDING, CATEGORY, CAUSAL_LINK) AS CHAPTER_PREFIX
FROM PROCESSING.ENCOUNTER_FINDINGS
WHERE CATEGORY != 'ruled_out';

-- All search paths + merge in one statement
CREATE OR REPLACE TABLE MATCHING.CANDIDATES_MERGED AS
WITH candidates_semantic AS (
    SELECT
        q.FILE_NAME, q.FINDING_SEQ, q.FINDING, q.CATEGORY, q.CAUSAL_LINK,
        r.ICD10_CODE,
        r.SHORT_DESCRIPTION AS DESCRIPTION,
        TRY_CAST(r.HCC_CATEGORY AS NUMBER) AS HCC_CATEGORY,
        r.METADATA$RANK AS CANDIDATE_RANK,
        'SEMANTIC' AS RETRIEVAL_PATH
    FROM MATCHING.SEARCH_QUERIES q,
    LATERAL CORTEX_SEARCH_BATCH(
        service_name => 'ICD10_CODING_APP.ICD10_REF.ICD10_SEARCH_SVC',
        query => q.SEARCH_QUERY,
        limit => $SEMANTIC_K
    ) AS r
),
candidates_category AS (
    SELECT
        q.FILE_NAME, q.FINDING_SEQ, q.FINDING, q.CATEGORY, q.CAUSAL_LINK,
        e.ICD10_CODE,
        e.SHORT_DESCRIPTION AS DESCRIPTION,
        e.HCC_CATEGORY,
        ROW_NUMBER() OVER (
            PARTITION BY q.FILE_NAME, q.FINDING_SEQ
            ORDER BY JAROWINKLER_SIMILARITY(LOWER(q.FINDING), LOWER(e.SHORT_DESCRIPTION)) DESC
        ) AS CANDIDATE_RANK,
        'CATEGORY' AS RETRIEVAL_PATH
    FROM MATCHING.SEARCH_QUERIES q
    JOIN ICD10_CODING_APP.ICD10_REF.ICD10_CODES_ENRICHED e ON e.ICD10_CODE LIKE q.CHAPTER_PREFIX || '%'
    WHERE q.CHAPTER_PREFIX IS NOT NULL
    QUALIFY CANDIDATE_RANK <= $CATEGORY_K
),
candidates_hcc AS (
    SELECT
        cs.FILE_NAME, cs.FINDING_SEQ, cs.FINDING, cs.CATEGORY, cs.CAUSAL_LINK,
        e.ICD10_CODE,
        e.SHORT_DESCRIPTION AS DESCRIPTION,
        e.HCC_CATEGORY,
        ROW_NUMBER() OVER (
            PARTITION BY cs.FILE_NAME, cs.FINDING_SEQ
            ORDER BY JAROWINKLER_SIMILARITY(LOWER(cs.FINDING), LOWER(e.SHORT_DESCRIPTION)) DESC
        ) AS CANDIDATE_RANK,
        'HCC_EXPANSION' AS RETRIEVAL_PATH
    FROM candidates_semantic cs
    JOIN ICD10_CODING_APP.ICD10_REF.ICD10_CODES_ENRICHED e
        ON e.HCC_CATEGORY = cs.HCC_CATEGORY
        AND e.ICD10_CODE != cs.ICD10_CODE
    WHERE cs.CANDIDATE_RANK = 1 AND cs.HCC_CATEGORY IS NOT NULL
    QUALIFY CANDIDATE_RANK <= $HCC_EXPANSION_K
),
all_candidates AS (
    SELECT FILE_NAME, FINDING_SEQ, FINDING, CATEGORY, CAUSAL_LINK, ICD10_CODE, DESCRIPTION, HCC_CATEGORY, CANDIDATE_RANK, RETRIEVAL_PATH,
           MATCHING.COMPUTE_PATH_SCORE(RETRIEVAL_PATH, CANDIDATE_RANK, $WEIGHT_SEMANTIC, $WEIGHT_CATEGORY, $WEIGHT_HCC) AS PATH_SCORE
    FROM candidates_semantic
    UNION ALL
    SELECT FILE_NAME, FINDING_SEQ, FINDING, CATEGORY, CAUSAL_LINK, ICD10_CODE, DESCRIPTION, HCC_CATEGORY, CANDIDATE_RANK, RETRIEVAL_PATH,
           MATCHING.COMPUTE_PATH_SCORE(RETRIEVAL_PATH, CANDIDATE_RANK, $WEIGHT_SEMANTIC, $WEIGHT_CATEGORY, $WEIGHT_HCC) AS PATH_SCORE
    FROM candidates_category
    UNION ALL
    SELECT FILE_NAME, FINDING_SEQ, FINDING, CATEGORY, CAUSAL_LINK, ICD10_CODE, DESCRIPTION, HCC_CATEGORY, CANDIDATE_RANK, RETRIEVAL_PATH,
           MATCHING.COMPUTE_PATH_SCORE(RETRIEVAL_PATH, CANDIDATE_RANK, $WEIGHT_SEMANTIC, $WEIGHT_CATEGORY, $WEIGHT_HCC) AS PATH_SCORE
    FROM candidates_hcc
),
scored AS (
    SELECT FILE_NAME, FINDING_SEQ, FINDING, CATEGORY, CAUSAL_LINK, ICD10_CODE,
           MAX(DESCRIPTION) AS DESCRIPTION,
           MAX(HCC_CATEGORY) AS HCC_CATEGORY,
           SUM(PATH_SCORE) AS COMBINED_SCORE,
           LISTAGG(DISTINCT RETRIEVAL_PATH, ',') AS FOUND_BY_PATHS,
           COUNT(DISTINCT RETRIEVAL_PATH) AS PATH_COUNT
    FROM all_candidates
    GROUP BY FILE_NAME, FINDING_SEQ, FINDING, CATEGORY, CAUSAL_LINK, ICD10_CODE
)
SELECT *,
    ROW_NUMBER() OVER (PARTITION BY FILE_NAME, FINDING_SEQ ORDER BY COMBINED_SCORE DESC) AS FINAL_RANK
FROM scored
QUALIFY FINAL_RANK <= $FINAL_TOP_N;

In [ ]:
%%sql -r matching_result
SET QUERY_TAG_VAL = 'experiment:matching|run:' || $RUN_ID;
ALTER SESSION SET QUERY_TAG = $QUERY_TAG_VAL;

CREATE OR REPLACE TABLE MATCHING.FINAL_ASSIGNMENTS_V2 AS
WITH candidate_json AS (
    SELECT cm.FILE_NAME, cm.FINDING_SEQ,
           ef.FINDING, ef.CATEGORY, ef.ACUITY, ef.LATERALITY, ef.CAUSAL_LINK, ef.SUPPORTING_QUOTE,
           ARRAY_AGG(OBJECT_CONSTRUCT('code', cm.ICD10_CODE, 'description', cm.DESCRIPTION, 'score', cm.COMBINED_SCORE))
               WITHIN GROUP (ORDER BY cm.FINAL_RANK) AS CANDIDATES_ARR
    FROM MATCHING.CANDIDATES_MERGED cm
    JOIN PROCESSING.ENCOUNTER_FINDINGS ef ON cm.FILE_NAME = ef.FILE_NAME AND cm.FINDING_SEQ = ef.FINDING_SEQ
    GROUP BY cm.FILE_NAME, cm.FINDING_SEQ, ef.FINDING, ef.CATEGORY, ef.ACUITY, ef.LATERALITY, ef.CAUSAL_LINK, ef.SUPPORTING_QUOTE
),
raw_assignments AS (
    SELECT *,
        ASSIGN_ICD10_CODE(FINDING, CATEGORY, ACUITY, LATERALITY, CAUSAL_LINK, SUPPORTING_QUOTE, CANDIDATES_ARR::VARCHAR) AS RAW_RESPONSE
    FROM candidate_json
)
SELECT
    FILE_NAME, FINDING_SEQ, FINDING, CATEGORY,
    TRY_PARSE_JSON(REGEXP_REPLACE(RAW_RESPONSE, '^```(json)?\\s*|\\s*```$', '')):code::VARCHAR AS ASSIGNED_CODE,
    TRY_PARSE_JSON(REGEXP_REPLACE(RAW_RESPONSE, '^```(json)?\\s*|\\s*```$', '')):description::VARCHAR AS ASSIGNED_DESCRIPTION,
    TRY_PARSE_JSON(REGEXP_REPLACE(RAW_RESPONSE, '^```(json)?\\s*|\\s*```$', '')):confidence::FLOAT AS CONFIDENCE,
    TRY_PARSE_JSON(REGEXP_REPLACE(RAW_RESPONSE, '^```(json)?\\s*|\\s*```$', '')):rationale::VARCHAR AS RATIONALE,
    CURRENT_TIMESTAMP() AS ASSIGNED_AT
FROM raw_assignments;

In [ ]:
%%sql -r eval_result
SET QUERY_TAG_VAL = 'experiment:evaluation|run:' || $RUN_ID;
ALTER SESSION SET QUERY_TAG = $QUERY_TAG_VAL;

USE SCHEMA EXPERIMENTS;

-- Extraction recall (LLM judge) - one row per distinct (chase, code)
CREATE OR REPLACE TABLE EXPERIMENTS.EVAL_EXTRACTION AS
WITH gt AS (
    SELECT DISTINCT
        UPPER(CHASE_ID) AS CHASE_ID,
        DIAGNOSIS_CODE AS GT_CODE
    FROM CHART_REVIEW_DB.RAW.AETNA_GROUND_TRUTH
),
findings_by_chase AS (
    SELECT
        UPPER(REGEXP_SUBSTR(FILE_NAME, 'eppmra[0-9]+', 1, 1, 'i')) AS CHASE_ID,
        LISTAGG(FINDING || ' (' || CATEGORY || ')', '; ') WITHIN GROUP (ORDER BY FINDING_SEQ) AS ALL_FINDINGS
    FROM PROCESSING.ENCOUNTER_FINDINGS
    GROUP BY CHASE_ID
)
SELECT gt.CHASE_ID, gt.GT_CODE,
    SNOWFLAKE.CORTEX.COMPLETE($EVAL_MODEL, CONCAT(
        'Does any finding in this list correspond to ICD-10 code ', gt.GT_CODE, '? ',
        'Findings: ', LEFT(f.ALL_FINDINGS, 3000),
        ' Answer ONLY with JSON: {"verdict": true/false, "rationale": "<brief>"}'
    )) AS RAW_VERDICT,
    TRY_PARSE_JSON(REGEXP_REPLACE(RAW_VERDICT, '^```(json)?\\s*|\\s*```$', '')):verdict::BOOLEAN AS VERDICT
FROM gt
JOIN findings_by_chase f ON gt.CHASE_ID = f.CHASE_ID;

-- Search recall
CREATE OR REPLACE TABLE EXPERIMENTS.EVAL_SEARCH AS
SELECT ee.CHASE_ID, ee.GT_CODE,
    EXISTS (
        SELECT 1 FROM MATCHING.CANDIDATES_MERGED cm
        WHERE UPPER(REGEXP_SUBSTR(cm.FILE_NAME, 'eppmra[0-9]+', 1, 1, 'i')) = ee.CHASE_ID
          AND REPLACE(cm.ICD10_CODE, '.', '') = REPLACE(ee.GT_CODE, '.', '')
    ) AS FOUND_IN_SEARCH,
    (SELECT MIN(cm.FINAL_RANK) FROM MATCHING.CANDIDATES_MERGED cm
     WHERE UPPER(REGEXP_SUBSTR(cm.FILE_NAME, 'eppmra[0-9]+', 1, 1, 'i')) = ee.CHASE_ID
       AND REPLACE(cm.ICD10_CODE, '.', '') = REPLACE(ee.GT_CODE, '.', '')
    ) AS BEST_RANK
FROM EXPERIMENTS.EVAL_EXTRACTION ee
WHERE ee.VERDICT = TRUE;

-- Full detail evaluation (exactly 2,112 rows = one per GT row)
CREATE OR REPLACE TABLE EXPERIMENTS.EVAL_FULL_DETAIL AS
WITH gt AS (
    SELECT
        UPPER(CHASE_ID) AS CHASE_ID,
        DIAGNOSIS_CODE AS GT_CODE
    FROM CHART_REVIEW_DB.RAW.AETNA_GROUND_TRUTH
),
assignments_deduped AS (
    SELECT
        UPPER(REGEXP_SUBSTR(FILE_NAME, 'eppmra[0-9]+', 1, 1, 'i')) AS CHASE_ID,
        REPLACE(ASSIGNED_CODE, '.', '') AS ASSIGNED_NODOT,
        MAX_BY(ASSIGNED_CODE, CONFIDENCE) AS ASSIGNED_CODE,
        MAX(CONFIDENCE) AS CONFIDENCE
    FROM MATCHING.FINAL_ASSIGNMENTS_V2
    WHERE ASSIGNED_CODE IS NOT NULL
    GROUP BY 1, 2
)
SELECT
    gt.CHASE_ID, gt.GT_CODE,
    COALESCE(ee.VERDICT, FALSE) AS EXTRACTED,
    COALESCE(es.FOUND_IN_SEARCH, FALSE) AS IN_SEARCH,
    fa.ASSIGNED_CODE,
    fa.CONFIDENCE,
    CASE
        WHEN fa.ASSIGNED_CODE IS NOT NULL THEN
            CASE WHEN fa.CONFIDENCE >= $CONFIDENCE_THRESHOLD THEN 'CORRECT_HIGH_CONF' ELSE 'CORRECT_LOW_CONF' END
        WHEN COALESCE(es.FOUND_IN_SEARCH, FALSE) THEN 'IN_CANDIDATES_NOT_MATCHED'
        WHEN COALESCE(ee.VERDICT, FALSE) THEN 'EXTRACTED_NOT_IN_CANDIDATES'
        ELSE 'NOT_EXTRACTED'
    END AS VERIFICATION_STATUS
FROM gt
LEFT JOIN EXPERIMENTS.EVAL_EXTRACTION ee ON gt.CHASE_ID = ee.CHASE_ID AND gt.GT_CODE = ee.GT_CODE
LEFT JOIN EXPERIMENTS.EVAL_SEARCH es ON gt.CHASE_ID = es.CHASE_ID AND gt.GT_CODE = es.GT_CODE
LEFT JOIN assignments_deduped fa ON gt.CHASE_ID = fa.CHASE_ID AND fa.ASSIGNED_NODOT = REPLACE(gt.GT_CODE, '.', '');

In [ ]:
%%sql -r logging_result
SET QUERY_TAG_VAL = 'experiment:logging|run:' || $RUN_ID;
ALTER SESSION SET QUERY_TAG = $QUERY_TAG_VAL;

-- Log the experiment run
INSERT INTO EXPERIMENTS.RUNS (RUN_ID, STRATEGY, MODEL, PARAMETERS_JSON)
SELECT $RUN_ID, $STRATEGY, $EXTRACTION_MODEL,
    OBJECT_CONSTRUCT(
        'extraction_model', $EXTRACTION_MODEL,
        'matching_model', $MATCHING_MODEL,
        'eval_model', $EVAL_MODEL,
        'semantic_k', $SEMANTIC_K,
        'category_k', $CATEGORY_K,
        'hcc_expansion_k', $HCC_EXPANSION_K,
        'final_top_n', $FINAL_TOP_N,
        'weight_semantic', $WEIGHT_SEMANTIC,
        'weight_category', $WEIGHT_CATEGORY,
        'weight_hcc', $WEIGHT_HCC,
        'confidence_threshold', $CONFIDENCE_THRESHOLD
    );

-- Log metrics
INSERT INTO EXPERIMENTS.METRICS (RUN_ID, METRIC_NAME, METRIC_VALUE)
SELECT $RUN_ID, 'extraction_recall',
    COUNT_IF(VERDICT = TRUE) / COUNT(*)::FLOAT
FROM EXPERIMENTS.EVAL_EXTRACTION
UNION ALL
SELECT $RUN_ID, 'search_recall',
    COUNT_IF(FOUND_IN_SEARCH) / COUNT(*)::FLOAT
FROM EXPERIMENTS.EVAL_SEARCH
UNION ALL
SELECT $RUN_ID, 'end_to_end_recall',
    COUNT_IF(VERIFICATION_STATUS IN ('CORRECT_HIGH_CONF','CORRECT_LOW_CONF')) / COUNT(*)::FLOAT
FROM EXPERIMENTS.EVAL_FULL_DETAIL
UNION ALL
SELECT $RUN_ID, 'correct_high_conf_pct',
    COUNT_IF(VERIFICATION_STATUS = 'CORRECT_HIGH_CONF') / COUNT(*)::FLOAT
FROM EXPERIMENTS.EVAL_FULL_DETAIL;

In [ ]:
%%sql -r accuracy_summary
SET QUERY_TAG_VAL = 'experiment:results|run:' || $RUN_ID;
ALTER SESSION SET QUERY_TAG = $QUERY_TAG_VAL;

-- Pipeline accuracy summary
SELECT
    VERIFICATION_STATUS,
    COUNT(*) AS CNT,
    ROUND(COUNT(*) / SUM(COUNT(*)) OVER () * 100, 1) AS PCT
FROM EXPERIMENTS.EVAL_FULL_DETAIL
GROUP BY VERIFICATION_STATUS
ORDER BY CNT DESC;

In [ ]:
%%sql -r funnel_metrics
-- Pipeline funnel: accuracy at each step (DISTINCT codes + ALL assignments)
SELECT 'DISTINCT' AS GRAIN, 'Step 1: Extraction' AS STEP,
    COUNT(*) AS TOTAL, COUNT_IF(EXTRACTED) AS PASSED,
    ROUND(COUNT_IF(EXTRACTED) * 100.0 / COUNT(*), 2) AS ACCURACY_PCT
FROM (SELECT DISTINCT CHASE_ID, GT_CODE, EXTRACTED FROM EXPERIMENTS.EVAL_FULL_DETAIL)
UNION ALL
SELECT 'DISTINCT', 'Step 2: Search',
    COUNT_IF(EXTRACTED), COUNT_IF(EXTRACTED AND IN_SEARCH),
    ROUND(COUNT_IF(EXTRACTED AND IN_SEARCH) * 100.0 / NULLIF(COUNT_IF(EXTRACTED), 0), 2)
FROM (SELECT DISTINCT CHASE_ID, GT_CODE, EXTRACTED, IN_SEARCH FROM EXPERIMENTS.EVAL_FULL_DETAIL)
UNION ALL
SELECT 'DISTINCT', 'Step 3: Matching',
    COUNT_IF(EXTRACTED AND IN_SEARCH),
    COUNT_IF(EXTRACTED AND IN_SEARCH AND VERIFICATION_STATUS IN ('CORRECT_HIGH_CONF','CORRECT_LOW_CONF')),
    ROUND(COUNT_IF(EXTRACTED AND IN_SEARCH AND VERIFICATION_STATUS IN ('CORRECT_HIGH_CONF','CORRECT_LOW_CONF')) * 100.0 / NULLIF(COUNT_IF(EXTRACTED AND IN_SEARCH), 0), 2)
FROM (SELECT DISTINCT CHASE_ID, GT_CODE, EXTRACTED, IN_SEARCH, VERIFICATION_STATUS FROM EXPERIMENTS.EVAL_FULL_DETAIL)
UNION ALL
SELECT 'DISTINCT', 'End-to-End',
    COUNT(*),
    COUNT_IF(VERIFICATION_STATUS IN ('CORRECT_HIGH_CONF','CORRECT_LOW_CONF')),
    ROUND(COUNT_IF(VERIFICATION_STATUS IN ('CORRECT_HIGH_CONF','CORRECT_LOW_CONF')) * 100.0 / COUNT(*), 2)
FROM (SELECT DISTINCT CHASE_ID, GT_CODE, VERIFICATION_STATUS FROM EXPERIMENTS.EVAL_FULL_DETAIL)
UNION ALL
SELECT 'ALL_ASSIGNMENTS', 'Step 1: Extraction',
    COUNT(*), COUNT_IF(EXTRACTED), ROUND(COUNT_IF(EXTRACTED) * 100.0 / COUNT(*), 2)
FROM EXPERIMENTS.EVAL_FULL_DETAIL
UNION ALL
SELECT 'ALL_ASSIGNMENTS', 'Step 2: Search',
    COUNT_IF(EXTRACTED), COUNT_IF(EXTRACTED AND IN_SEARCH),
    ROUND(COUNT_IF(EXTRACTED AND IN_SEARCH) * 100.0 / NULLIF(COUNT_IF(EXTRACTED), 0), 2)
FROM EXPERIMENTS.EVAL_FULL_DETAIL
UNION ALL
SELECT 'ALL_ASSIGNMENTS', 'Step 3: Matching',
    COUNT_IF(EXTRACTED AND IN_SEARCH),
    COUNT_IF(EXTRACTED AND IN_SEARCH AND VERIFICATION_STATUS IN ('CORRECT_HIGH_CONF','CORRECT_LOW_CONF')),
    ROUND(COUNT_IF(EXTRACTED AND IN_SEARCH AND VERIFICATION_STATUS IN ('CORRECT_HIGH_CONF','CORRECT_LOW_CONF')) * 100.0 / NULLIF(COUNT_IF(EXTRACTED AND IN_SEARCH), 0), 2)
FROM EXPERIMENTS.EVAL_FULL_DETAIL
UNION ALL
SELECT 'ALL_ASSIGNMENTS', 'End-to-End',
    COUNT(*),
    COUNT_IF(VERIFICATION_STATUS IN ('CORRECT_HIGH_CONF','CORRECT_LOW_CONF')),
    ROUND(COUNT_IF(VERIFICATION_STATUS IN ('CORRECT_HIGH_CONF','CORRECT_LOW_CONF')) * 100.0 / COUNT(*), 2)
FROM EXPERIMENTS.EVAL_FULL_DETAIL
ORDER BY GRAIN, STEP;

In [ ]:
%%sql -r ai_costs
-- AI credits per pipeline step for this run
-- Note: CORTEX_AI_FUNCTIONS_USAGE_HISTORY has up to 2 hours of latency
SELECT
    QUERY_TAG,
    FUNCTION_NAME,
    MODEL_NAME,
    COUNT(DISTINCT QUERY_ID) AS AI_CALLS,
    SUM(CREDITS) AS AI_CREDITS
FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_FUNCTIONS_USAGE_HISTORY
WHERE QUERY_TAG LIKE '%run:' || $RUN_ID
    AND START_TIME >= DATEADD('day', -7, CURRENT_TIMESTAMP())
GROUP BY QUERY_TAG, FUNCTION_NAME, MODEL_NAME
ORDER BY QUERY_TAG;

In [ ]:
%%sql -r reset_tag
-- Reset query tag
ALTER SESSION SET QUERY_TAG = '';

In [ ]:
%%sql -r credits_per_encounter
-- AI credits per encounter
WITH encounter_count AS (
    SELECT COUNT(*) AS NUM_ENCOUNTERS
    FROM PROCESSING.DOCUMENT_SECTIONS
),
total_credits AS (
    SELECT COALESCE(SUM(CREDITS), 0) AS TOTAL_AI_CREDITS
    FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AI_FUNCTIONS_USAGE_HISTORY
    WHERE QUERY_TAG LIKE '%run:' || $RUN_ID
        AND START_TIME >= DATEADD('day', -7, CURRENT_TIMESTAMP())
)
SELECT
    e.NUM_ENCOUNTERS,
    t.TOTAL_AI_CREDITS,
    ROUND(t.TOTAL_AI_CREDITS / NULLIF(e.NUM_ENCOUNTERS, 0), 6) AS CREDITS_PER_ENCOUNTER
FROM encounter_count e, total_credits t;

In [ ]:
%%sql -r extra_diagnoses
-- Extra diagnoses: codes assigned by pipeline but NOT in ground truth
WITH pipeline_codes AS (
    SELECT DISTINCT
        UPPER(REGEXP_SUBSTR(FILE_NAME, 'eppmra[0-9]+', 1, 1, 'i')) AS CHASE_ID,
        REPLACE(ASSIGNED_CODE, '.', '') AS CODE_NODOT,
        ASSIGNED_CODE,
        ASSIGNED_DESCRIPTION,
        CONFIDENCE
    FROM MATCHING.FINAL_ASSIGNMENTS_V2
    WHERE ASSIGNED_CODE IS NOT NULL
),
gt_codes AS (
    SELECT DISTINCT
        UPPER(CHASE_ID) AS CHASE_ID,
        REPLACE(DIAGNOSIS_CODE, '.', '') AS CODE_NODOT
    FROM CHART_REVIEW_DB.RAW.AETNA_GROUND_TRUTH
)
SELECT
    COUNT(*) AS EXTRA_DIAGNOSES_IDENTIFIED,
    COUNT(DISTINCT p.CHASE_ID) AS ENCOUNTERS_WITH_EXTRAS,
    ROUND(COUNT(*) * 1.0 / NULLIF(COUNT(DISTINCT p.CHASE_ID), 0), 1) AS AVG_EXTRAS_PER_ENCOUNTER
FROM pipeline_codes p
LEFT JOIN gt_codes g ON p.CHASE_ID = g.CHASE_ID AND p.CODE_NODOT = g.CODE_NODOT
WHERE g.CODE_NODOT IS NULL;

In [ ]:
%%sql -r hedis_tagging
-- Tag all GT and pipeline codes with HEDIS measure relevance, then compute HEDIS-only metrics
CREATE OR REPLACE TEMPORARY TABLE EXPERIMENTS.HEDIS_ANALYSIS AS
WITH all_codes AS (
    -- Ground truth codes
    SELECT DISTINCT GT_CODE AS ICD10_CODE FROM EXPERIMENTS.EVAL_FULL_DETAIL
    UNION
    -- Extra pipeline codes not in GT
    SELECT DISTINCT REPLACE(ASSIGNED_CODE, '.', '') AS ICD10_CODE
    FROM MATCHING.FINAL_ASSIGNMENTS_V2
    WHERE ASSIGNED_CODE IS NOT NULL
),
hedis_tagged AS (
    SELECT
        ICD10_CODE,
        SNOWFLAKE.CORTEX.COMPLETE('claude-haiku-4-5', CONCAT(
            'Is ICD-10 code ', ICD10_CODE, ' associated with any common HEDIS quality measure? ',
            'If yes, respond ONLY with JSON: {"hedis": true, "measure": "<abbreviation: e.g. CDC, CBP, BCS, CCS, SPD, AMR, PBH, DSF>", "measure_name": "<full name>"}. ',
            'If no, respond ONLY with: {"hedis": false}'
        )) AS RAW_RESPONSE,
        TRY_PARSE_JSON(REGEXP_REPLACE(RAW_RESPONSE, '^```(json)?\\s*|\\s*```$', '')) AS PARSED
    FROM all_codes
)
SELECT
    ICD10_CODE,
    PARSED:hedis::BOOLEAN AS IS_HEDIS,
    PARSED:measure::VARCHAR AS HEDIS_MEASURE,
    PARSED:measure_name::VARCHAR AS HEDIS_MEASURE_NAME
FROM hedis_tagged;

In [ ]:
%%sql -r hedis_accuracy
-- HEDIS accuracy: funnel metrics for HEDIS-related GT codes only
WITH hedis_gt AS (
    SELECT e.*
    FROM EXPERIMENTS.EVAL_FULL_DETAIL e
    JOIN EXPERIMENTS.HEDIS_ANALYSIS h ON REPLACE(e.GT_CODE, '.', '') = h.ICD10_CODE AND h.IS_HEDIS = TRUE
)
SELECT
    'HEDIS GT Codes' AS CATEGORY,
    COUNT(*) AS TOTAL,
    COUNT_IF(EXTRACTED) AS EXTRACTED,
    COUNT_IF(EXTRACTED AND IN_SEARCH) AS IN_SEARCH,
    COUNT_IF(EXTRACTED AND IN_SEARCH AND VERIFICATION_STATUS IN ('CORRECT_HIGH_CONF','CORRECT_LOW_CONF')) AS MATCHED,
    ROUND(COUNT_IF(VERIFICATION_STATUS IN ('CORRECT_HIGH_CONF','CORRECT_LOW_CONF')) * 100.0 / COUNT(*), 2) AS END_TO_END_PCT,
    COUNT_IF(VERIFICATION_STATUS = 'NOT_EXTRACTED') AS MISSED_AT_EXTRACTION,
    COUNT_IF(VERIFICATION_STATUS = 'EXTRACTED_NOT_IN_CANDIDATES') AS MISSED_AT_SEARCH,
    COUNT_IF(VERIFICATION_STATUS = 'IN_CANDIDATES_NOT_MATCHED') AS MISSED_AT_MATCHING
FROM hedis_gt
UNION ALL
SELECT
    'Non-HEDIS GT Codes',
    COUNT(*),
    COUNT_IF(EXTRACTED),
    COUNT_IF(EXTRACTED AND IN_SEARCH),
    COUNT_IF(EXTRACTED AND IN_SEARCH AND VERIFICATION_STATUS IN ('CORRECT_HIGH_CONF','CORRECT_LOW_CONF')),
    ROUND(COUNT_IF(VERIFICATION_STATUS IN ('CORRECT_HIGH_CONF','CORRECT_LOW_CONF')) * 100.0 / COUNT(*), 2),
    COUNT_IF(VERIFICATION_STATUS = 'NOT_EXTRACTED'),
    COUNT_IF(VERIFICATION_STATUS = 'EXTRACTED_NOT_IN_CANDIDATES'),
    COUNT_IF(VERIFICATION_STATUS = 'IN_CANDIDATES_NOT_MATCHED')
FROM EXPERIMENTS.EVAL_FULL_DETAIL e
LEFT JOIN EXPERIMENTS.HEDIS_ANALYSIS h ON REPLACE(e.GT_CODE, '.', '') = h.ICD10_CODE AND h.IS_HEDIS = TRUE
WHERE h.ICD10_CODE IS NULL;

In [ ]:
%%sql -r hedis_summary
-- HEDIS summary: captured, missed, and new suggestions
WITH gt_hedis AS (
    SELECT DISTINCT e.CHASE_ID, e.GT_CODE, e.VERIFICATION_STATUS, h.HEDIS_MEASURE, h.HEDIS_MEASURE_NAME
    FROM EXPERIMENTS.EVAL_FULL_DETAIL e
    JOIN EXPERIMENTS.HEDIS_ANALYSIS h ON REPLACE(e.GT_CODE, '.', '') = h.ICD10_CODE AND h.IS_HEDIS = TRUE
),
extra_hedis AS (
    SELECT DISTINCT
        UPPER(REGEXP_SUBSTR(f.FILE_NAME, 'eppmra[0-9]+', 1, 1, 'i')) AS CHASE_ID,
        REPLACE(f.ASSIGNED_CODE, '.', '') AS CODE_NODOT,
        f.ASSIGNED_CODE,
        h.HEDIS_MEASURE,
        h.HEDIS_MEASURE_NAME
    FROM MATCHING.FINAL_ASSIGNMENTS_V2 f
    JOIN EXPERIMENTS.HEDIS_ANALYSIS h ON REPLACE(f.ASSIGNED_CODE, '.', '') = h.ICD10_CODE AND h.IS_HEDIS = TRUE
    WHERE f.ASSIGNED_CODE IS NOT NULL
      AND NOT EXISTS (
          SELECT 1 FROM CHART_REVIEW_DB.RAW.AETNA_GROUND_TRUTH g
          WHERE UPPER(g.CHASE_ID) = UPPER(REGEXP_SUBSTR(f.FILE_NAME, 'eppmra[0-9]+', 1, 1, 'i'))
            AND REPLACE(g.DIAGNOSIS_CODE, '.', '') = REPLACE(f.ASSIGNED_CODE, '.', '')
      )
)
SELECT 'HEDIS Captured (correct)' AS METRIC,
    COUNT(*) AS COUNT,
    LISTAGG(DISTINCT HEDIS_MEASURE, ', ') WITHIN GROUP (ORDER BY HEDIS_MEASURE) AS MEASURES
FROM gt_hedis WHERE VERIFICATION_STATUS IN ('CORRECT_HIGH_CONF','CORRECT_LOW_CONF')
UNION ALL
SELECT 'HEDIS Missed (in GT, not matched)',
    COUNT(*),
    LISTAGG(DISTINCT HEDIS_MEASURE, ', ') WITHIN GROUP (ORDER BY HEDIS_MEASURE)
FROM gt_hedis WHERE VERIFICATION_STATUS NOT IN ('CORRECT_HIGH_CONF','CORRECT_LOW_CONF')
UNION ALL
SELECT 'HEDIS New Suggestions (extra, not in GT)',
    COUNT(*),
    LISTAGG(DISTINCT HEDIS_MEASURE, ', ') WITHIN GROUP (ORDER BY HEDIS_MEASURE)
FROM extra_hedis;

In [ ]:
%%sql -r batch_counts
-- Row counts per batch (from pipeline inputs)
SELECT
    CASE
        WHEN FILE_NAME LIKE 'encounters/batch_01_aetna/%' THEN 'batch_01_aetna'
        WHEN FILE_NAME LIKE 'encounters/batch_01_asrs/%'  THEN 'batch_01_asrs'
        ELSE 'test_encounters'
    END AS BATCH,
    COUNT(*) AS ENCOUNTER_ROWS,
    COUNT(DISTINCT FILE_NAME) AS DISTINCT_FILES,
    COUNT(DISTINCT UPPER(REGEXP_SUBSTR(FILE_NAME, 'eppmra[0-9]+', 1, 1, 'i'))) AS DISTINCT_CHASES
FROM ICD10_CODING_APP.PROCESSING.DOCUMENT_SECTIONS
GROUP BY BATCH
ORDER BY BATCH

In [ ]:
%%sql -r perf_by_batch
-- Pipeline funnel by batch (encounter-level throughput + GT recall)
WITH batch_map AS (
    SELECT DISTINCT
        FILE_NAME,
        UPPER(REGEXP_SUBSTR(FILE_NAME, 'eppmra[0-9]+', 1, 1, 'i')) AS CHASE_ID,
        CASE
            WHEN FILE_NAME LIKE 'encounters/batch_01_aetna/%' THEN 'batch_01_aetna'
            WHEN FILE_NAME LIKE 'encounters/batch_01_asrs/%'  THEN 'batch_01_asrs'
            ELSE 'test_encounters'
        END AS BATCH
    FROM ICD10_CODING_APP.PROCESSING.DOCUMENT_SECTIONS
),
-- Step 0: Input encounters
input_counts AS (
    SELECT BATCH,
        COUNT(*) AS ENCOUNTER_ROWS,
        COUNT(DISTINCT CHASE_ID) AS DISTINCT_CHASES
    FROM batch_map
    GROUP BY BATCH
),
-- Step 1: Extraction output
extraction_counts AS (
    SELECT b.BATCH,
        COUNT(DISTINCT b.CHASE_ID) AS CHASES_WITH_FINDINGS,
        COUNT(*) AS TOTAL_FINDINGS
    FROM ICD10_CODING_APP.PROCESSING.ENCOUNTER_FINDINGS ef
    JOIN batch_map b ON ef.FILE_NAME = b.FILE_NAME
    GROUP BY b.BATCH
),
-- Step 2: Search output
search_counts AS (
    SELECT b.BATCH,
        COUNT(DISTINCT b.CHASE_ID) AS CHASES_WITH_CANDIDATES,
        COUNT(DISTINCT b.CHASE_ID || '|' || cm.FINDING_SEQ) AS FINDINGS_WITH_CANDIDATES
    FROM ICD10_CODING_APP.MATCHING.CANDIDATES_MERGED cm
    JOIN batch_map b ON cm.FILE_NAME = b.FILE_NAME
    GROUP BY b.BATCH
),
-- Step 3: Matching output
matching_counts AS (
    SELECT b.BATCH,
        COUNT(DISTINCT b.CHASE_ID) AS CHASES_WITH_ASSIGNMENTS,
        COUNT(*) AS TOTAL_ASSIGNMENTS,
        COUNT_IF(fa.CONFIDENCE >= 0.75) AS HIGH_CONF_ASSIGNMENTS
    FROM ICD10_CODING_APP.MATCHING.FINAL_ASSIGNMENTS_V2 fa
    JOIN batch_map b ON fa.FILE_NAME = b.FILE_NAME
    WHERE fa.ASSIGNED_CODE IS NOT NULL
    GROUP BY b.BATCH
),
-- GT recall (only meaningful where GT has codes)
gt_recall AS (
    SELECT b.BATCH,
        COUNT(*) AS GT_CODES,
        COUNT_IF(e.VERIFICATION_STATUS IN ('CORRECT_HIGH_CONF','CORRECT_LOW_CONF')) AS GT_MATCHED,
        ROUND(COUNT_IF(e.VERIFICATION_STATUS IN ('CORRECT_HIGH_CONF','CORRECT_LOW_CONF')) * 100.0 / NULLIF(COUNT(*), 0), 2) AS RECALL_PCT
    FROM ICD10_CODING_APP.EXPERIMENTS.EVAL_FULL_DETAIL e
    JOIN batch_map b ON e.CHASE_ID = b.CHASE_ID
    GROUP BY b.BATCH
)
SELECT
    i.BATCH,
    i.ENCOUNTER_ROWS,
    i.DISTINCT_CHASES,
    COALESCE(ex.CHASES_WITH_FINDINGS, 0) AS CHASES_EXTRACTED,
    COALESCE(ex.TOTAL_FINDINGS, 0) AS FINDINGS_EXTRACTED,
    COALESCE(s.CHASES_WITH_CANDIDATES, 0) AS CHASES_SEARCHED,
    COALESCE(s.FINDINGS_WITH_CANDIDATES, 0) AS FINDINGS_WITH_CANDIDATES,
    COALESCE(m.CHASES_WITH_ASSIGNMENTS, 0) AS CHASES_MATCHED,
    COALESCE(m.TOTAL_ASSIGNMENTS, 0) AS CODES_ASSIGNED,
    COALESCE(m.HIGH_CONF_ASSIGNMENTS, 0) AS HIGH_CONF_CODES,
    COALESCE(g.GT_CODES, 0) AS GT_CODES,
    COALESCE(g.GT_MATCHED, 0) AS GT_MATCHED,
    g.RECALL_PCT AS GT_RECALL_PCT
FROM input_counts i
LEFT JOIN extraction_counts ex ON i.BATCH = ex.BATCH
LEFT JOIN search_counts s ON i.BATCH = s.BATCH
LEFT JOIN matching_counts m ON i.BATCH = m.BATCH
LEFT JOIN gt_recall g ON i.BATCH = g.BATCH
ORDER BY i.BATCH

In [ ]:
%%sql -r asrs_per_chart
-- Findings assigned per chart for ASRS batch
SELECT
    UPPER(REGEXP_SUBSTR(FILE_NAME, 'eppmra[0-9]+', 1, 1, 'i')) AS CHASE_ID,
    COUNT(*) AS CODES_ASSIGNED,
    COUNT_IF(CONFIDENCE >= 0.75) AS HIGH_CONF,
    ROUND(AVG(CONFIDENCE), 3) AS AVG_CONFIDENCE
FROM ICD10_CODING_APP.MATCHING.FINAL_ASSIGNMENTS_V2
WHERE FILE_NAME LIKE 'encounters/batch_01_asrs/%'
  AND ASSIGNED_CODE IS NOT NULL
GROUP BY CHASE_ID
ORDER BY CODES_ASSIGNED DESC

In [ ]:
%%sql -r asrs_top_codes
-- Top diagnosis codes assigned for ASRS batch
SELECT
    ASSIGNED_CODE,
    ASSIGNED_DESCRIPTION,
    COUNT(*) AS TIMES_ASSIGNED,
    COUNT(DISTINCT UPPER(REGEXP_SUBSTR(FILE_NAME, 'eppmra[0-9]+', 1, 1, 'i'))) AS CHASES,
    ROUND(AVG(CONFIDENCE), 3) AS AVG_CONFIDENCE
FROM ICD10_CODING_APP.MATCHING.FINAL_ASSIGNMENTS_V2
WHERE FILE_NAME LIKE 'encounters/batch_01_asrs/%'
  AND ASSIGNED_CODE IS NOT NULL
GROUP BY ASSIGNED_CODE, ASSIGNED_DESCRIPTION
ORDER BY TIMES_ASSIGNED DESC
LIMIT 30

In [ ]:
%%sql -r hcc_analysis
-- HCC analysis: GT codes vs pipeline codes joined to ICD10 reference
-- Goal 1: HCC code accuracy
-- Goal 2: Accuracy filtered to HCC-only (non-null HCC) codes
-- Goal 3: Aetna should have HCC codes, ASRS should not
WITH batch_map AS (
    SELECT DISTINCT
        UPPER(REGEXP_SUBSTR(FILE_NAME, 'eppmra[0-9]+', 1, 1, 'i')) AS CHASE_ID,
        CASE
            WHEN FILE_NAME LIKE 'encounters/batch_01_aetna/%' THEN 'batch_01_aetna'
            WHEN FILE_NAME LIKE 'encounters/batch_01_asrs/%'  THEN 'batch_01_asrs'
            ELSE 'test_encounters'
        END AS BATCH
    FROM ICD10_CODING_APP.PROCESSING.DOCUMENT_SECTIONS
),
detail AS (
    SELECT
        b.BATCH,
        e.CHASE_ID,
        e.GT_CODE,
        e.ASSIGNED_CODE,
        e.CONFIDENCE,
        e.VERIFICATION_STATUS,
        ref_gt.HCC_CATEGORY   AS GT_HCC,
        ref_gt.HCC_DESCRIPTION AS GT_HCC_DESC,
        ref_pred.HCC_CATEGORY  AS PRED_HCC,
        ref_pred.HCC_DESCRIPTION AS PRED_HCC_DESC
    FROM ICD10_CODING_APP.EXPERIMENTS.EVAL_FULL_DETAIL e
    JOIN batch_map b ON e.CHASE_ID = b.CHASE_ID
    LEFT JOIN ICD10_CODING_APP.ICD10_REF.ICD10_CODES_ENRICHED ref_gt
        ON ref_gt.ICD10_CODE = e.GT_CODE
    LEFT JOIN ICD10_CODING_APP.ICD10_REF.ICD10_CODES_ENRICHED ref_pred
        ON ref_pred.ICD10_CODE = REPLACE(e.ASSIGNED_CODE, '.', '')
)
-- Summary by batch
SELECT
    BATCH,
    COUNT(*) AS TOTAL_GT_ROWS,

    -- GT HCC breakdown
    COUNT_IF(GT_HCC IS NOT NULL) AS GT_HAS_HCC,
    COUNT_IF(GT_HCC IS NULL AND GT_CODE IS NOT NULL) AS GT_NO_HCC,
    COUNT_IF(GT_CODE IS NULL) AS GT_CODE_NULL,

    -- Pipeline HCC breakdown
    COUNT_IF(PRED_HCC IS NOT NULL) AS PRED_HAS_HCC,
    COUNT_IF(PRED_HCC IS NULL AND ASSIGNED_CODE IS NOT NULL AND ASSIGNED_CODE != 'NONE') AS PRED_NO_HCC,

    -- Goal 1: HCC match accuracy (when both GT and pred have HCC)
    COUNT_IF(GT_HCC IS NOT NULL AND PRED_HCC IS NOT NULL AND GT_HCC = PRED_HCC) AS HCC_EXACT_MATCH,
    ROUND(COUNT_IF(GT_HCC IS NOT NULL AND PRED_HCC IS NOT NULL AND GT_HCC = PRED_HCC) * 100.0
        / NULLIF(COUNT_IF(GT_HCC IS NOT NULL), 0), 2) AS HCC_MATCH_PCT,

    -- Goal 2: End-to-end recall filtered to HCC-only GT codes
    COUNT_IF(GT_HCC IS NOT NULL AND VERIFICATION_STATUS IN ('CORRECT_HIGH_CONF','CORRECT_LOW_CONF')) AS HCC_GT_MATCHED,
    ROUND(COUNT_IF(GT_HCC IS NOT NULL AND VERIFICATION_STATUS IN ('CORRECT_HIGH_CONF','CORRECT_LOW_CONF')) * 100.0
        / NULLIF(COUNT_IF(GT_HCC IS NOT NULL), 0), 2) AS HCC_RECALL_PCT,

    -- For comparison: recall on non-HCC GT codes
    COUNT_IF(GT_HCC IS NULL AND GT_CODE IS NOT NULL AND VERIFICATION_STATUS IN ('CORRECT_HIGH_CONF','CORRECT_LOW_CONF')) AS NON_HCC_GT_MATCHED,
    ROUND(COUNT_IF(GT_HCC IS NULL AND GT_CODE IS NOT NULL AND VERIFICATION_STATUS IN ('CORRECT_HIGH_CONF','CORRECT_LOW_CONF')) * 100.0
        / NULLIF(COUNT_IF(GT_HCC IS NULL AND GT_CODE IS NOT NULL), 0), 2) AS NON_HCC_RECALL_PCT

FROM detail
GROUP BY BATCH
ORDER BY BATCH

In [ ]:
%%sql -r asrs_hcc_breakdown
-- HCC vs non-HCC codes assigned by pipeline: aetna vs asrs
SELECT
    CASE
        WHEN fa.FILE_NAME LIKE 'encounters/batch_01_aetna/%' THEN 'batch_01_aetna'
        WHEN fa.FILE_NAME LIKE 'encounters/batch_01_asrs/%'  THEN 'batch_01_asrs'
        ELSE 'test_encounters'
    END AS BATCH,
    CASE WHEN ref.HCC_CATEGORY IS NOT NULL THEN 'HCC (chronic)' ELSE 'Non-HCC' END AS CODE_TYPE,
    COUNT(*) AS CODES_ASSIGNED,
    COUNT(DISTINCT UPPER(REGEXP_SUBSTR(fa.FILE_NAME, 'eppmra[0-9]+', 1, 1, 'i'))) AS CHASES,
    ROUND(AVG(fa.CONFIDENCE), 3) AS AVG_CONFIDENCE
FROM ICD10_CODING_APP.MATCHING.FINAL_ASSIGNMENTS_V2 fa
LEFT JOIN ICD10_CODING_APP.ICD10_REF.ICD10_CODES_ENRICHED ref
    ON ref.ICD10_CODE = REPLACE(fa.ASSIGNED_CODE, '.', '')
WHERE fa.ASSIGNED_CODE IS NOT NULL
  AND fa.ASSIGNED_CODE != 'NONE'
  AND fa.FILE_NAME LIKE 'encounters/batch_01_%'
GROUP BY BATCH, CODE_TYPE
ORDER BY BATCH, CODE_TYPE

In [ ]:
%%sql -r asrs_hcc_detail
-- ASRS: HCC codes assigned with clinical evidence and full encounter text
SELECT
    UPPER(REGEXP_SUBSTR(fa.FILE_NAME, 'eppmra[0-9]+', 1, 1, 'i')) AS CHASE_ID,
    fa.ASSIGNED_CODE,
    fa.ASSIGNED_DESCRIPTION,
    ref.HCC_CATEGORY,
    ref.HCC_DESCRIPTION,
    fa.CONFIDENCE,
    fa.RATIONALE,
    ef.FINDING,
    ef.CATEGORY,
    ef.SUPPORTING_QUOTE,
    ds.PARSED_TEXT AS FULL_ENCOUNTER_TEXT
FROM ICD10_CODING_APP.MATCHING.FINAL_ASSIGNMENTS_V2 fa
JOIN ICD10_CODING_APP.ICD10_REF.ICD10_CODES_ENRICHED ref
    ON ref.ICD10_CODE = REPLACE(fa.ASSIGNED_CODE, '.', '')
    AND ref.HCC_CATEGORY IS NOT NULL
LEFT JOIN ICD10_CODING_APP.PROCESSING.ENCOUNTER_FINDINGS ef
    ON ef.FILE_NAME = fa.FILE_NAME AND ef.FINDING_SEQ = fa.FINDING_SEQ
LEFT JOIN ICD10_CODING_APP.PROCESSING.DOCUMENT_SECTIONS ds
    ON ds.FILE_NAME = fa.FILE_NAME
WHERE fa.FILE_NAME LIKE 'encounters/batch_01_asrs/%'
  AND fa.ASSIGNED_CODE IS NOT NULL
  AND fa.ASSIGNED_CODE != 'NONE'
ORDER BY ref.HCC_CATEGORY, fa.CONFIDENCE DESC